# Demo 2 - Multi-Agent Team Interaction (Agent Collaboration)
By: [Lior Gazit](https://github.com/LiorGazit).  
Repo: [Agents-Over-The-Weekend](https://github.com/PacktPublishing/Agents-Over-The-Weekend/tree/main/Lior_Gazit/workshop_september_2025/)   
Running LLMs locally for free: This code leverages [`LLMPop`](https://pypi.org/project/llmpop/) that is dedicated to spinning up local or remote LLMs in a unified and modular syntax.  

<a target="_blank" href="https://colab.research.google.com/github/PacktPublishing/Agents-Over-The-Weekend/blob/main/Lior_Gazit/workshop_september_2025/codes_for_Lior_Bootcamp_talk_sept2025_demo2.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a> (pick a GPU Colab session for fastest computing)  

```
Disclaimer: The content and ideas presented in this notebook are solely those of the author, Lior Gazit, and do not represent the views or intellectual property of the author's employer.
```

Installing:

In [1]:
%pip -q install llmpop
%pip -q install sentence-transformers faiss-cpu langchain tiktoken langsmith langchain_openai autogen-agentchat autogen-ext[openai] agent-framework

from llmpop import install_ollama_deps
install_ollama_deps()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.9/90.9 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.6/321.6 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.4/191.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**Imports:**

In [2]:
import os
import requests

In [3]:
from llmpop import init_llm
from langchain_core.prompts import ChatPromptTemplate

coder = init_llm(model="CodeLlama", provider="ollama", verbose=False)
reviewer = init_llm(model="llama3.2:1b", provider="ollama", verbose=False)

task = "Write a Python function to check if a number is prime."

# ---- Agent A: Coder ----
coder_prompt = ChatPromptTemplate.from_template(
    """
System: You are a coding agent. Output *code only*.
User task: {task}
"""
)

coder_result = (coder_prompt | coder).invoke({"task": task})
coder_code = getattr(coder_result, "content", str(coder_result)).strip()

# ---- Agent B: Reviewer ----
reviewer_prompt = ChatPromptTemplate.from_template(
    """
System: You are a strict code reviewer for Python.
Given the code below, evaluate for:
- Correctness and edge cases
- Time complexity reasonableness
- Readability (within reason, given constraints)
If changes are needed, provide a brief rationale followed by a *fully corrected* version.
If it's good as-is, say "APPROVED" and explain briefly why.

Code:
```python
{code}
```
"""
)

review_result = (reviewer_prompt | reviewer).invoke({"code": coder_code})
review_text = getattr(review_result, "content", str(review_result)).strip()

print("\n=== Coder output ===\n")
print(coder_code)

print("\n=== Reviewer feedback ===\n")
print(review_text)



=== Coder output ===

def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True

=== Reviewer feedback ===

APPROVED

The code provided is correct in terms of correctness and edge cases. It checks if a number is prime by iterating up to the square root of the number and checking for divisibility. The code also handles negative numbers and edge cases correctly.

However, it's worth noting that the function does not handle non-integer inputs. If the input is not an integer, the function may throw a TypeError or return incorrect results.

Here is a fully corrected version:

```python
import math

def is_prime(n):
    """
    Check if a number is prime.

    Args:
        n (int): The number to check.

    Returns:
        bool: True if the number is prime, False otherwise.
    """
    if not isinstance(n, int):
        raise TypeError("Input must be an integer.")
    if n < 2:
        retu

Now, here is an example using Microsoft Agent Framework:

In [4]:
import os
import asyncio
# In Colab, use getpass to securely prompt for your API key
from getpass import getpass
from agent_framework.openai import OpenAIChatClient
import openai

openai.api_key = getpass("Paste your OpenAI API key: ")

# 1. Import the agent classes and the OpenAI client


async def multi_agent_demo():
    # 2. Configure your OpenAI API key
    api_key = openai.api_key
    if not api_key:
        openai.api_key = getpass("Paste your OpenAI API key: ")

    # 3. Create the OpenAI model client
    client = OpenAIChatClient(model_id="gpt-5.2-2025-12-11", api_key=api_key)

    # 4. Instantiate two LLM agents with distinct roles
    coder = client.as_agent(
        name="Coder",
        instructions="You are a Python coding assistant. Produce only working code."
    )

    reviewer = client.as_agent(
        name="Reviewer",
        instructions="You are a code reviewer. Point out bugs or edge cases."
    )

    # 5. Coder agent writes a function
    code_task = "Write a Python function `is_prime(n)` that returns True if `n` is prime."
    code_result = await coder.run(code_task)
    print("=== Coder's Output ===\n")
    print(code_result.text)

    # 6. Reviewer agent critiques the code
    review_result = await reviewer.run(
        f"Review the following code for correctness and edge cases:\n\n{code_result.text}"
    )
    print("\n=== Reviewer's Feedback ===\n")
    print(review_result.text)

# 8. Execute the multi‑agent demo
await multi_agent_demo()

Paste your OpenAI API key: ··········
=== Coder's Output ===

```python
def is_prime(n: int) -> bool:
    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False

    i = 5
    step = 2
    while i * i <= n:
        if n % i == 0:
            return False
        i += step
        step = 6 - step  # checks 6k±1
    return True
```

=== Reviewer's Feedback ===

- **Correctness (core logic):**  
  This is a standard optimized trial-division primality test. After excluding `n < 2`, handling `2`/`3`, and removing multiples of `2` or `3`, it checks divisors of the form `6k ± 1` (5, 7, 11, 13, …) up to `sqrt(n)`. The `step = 6 - step` toggling `+2, +4, +2, +4...` is correct.

- **Edge cases covered well:**
  - `n = 0, 1` → `False`
  - `n = 2, 3` → `True`
  - Even numbers and multiples of 3 → `False`
  - Perfect squares like `25, 49, 121` → correctly `False` because `i*i <= n` includes `i == sqrt(n)`.

- **Potential issues /